In [2]:
import torch
import torch.nn as nn

In [3]:
# 1. Data Preparation
text = "I love machine learning and I love deep learning"
words = text.split()

vocab = sorted(set(words))

word_to_ix = {w: i for i, w in enumerate(vocab)}
ix_to_word = {i: w for w, i in word_to_ix.items()}

In [4]:
# 2. Generate Training Data
data = []

for i in range(1, len(words) - 1):
    context = [words[i - 1], words[i + 1]]
    target = words[i]
    data.append((context, target))

In [5]:
# 3. Define CBOW Model
class CBOW(nn.Module):

    def __init__(self, vocab_size, embedding_dim):
        super().__init__()

        self.embeddings = nn.Embedding(
            vocab_size, embedding_dim
        )

        self.linear = nn.Linear(
            embedding_dim, vocab_size
        )

    def forward(self, context):
        embeds = self.embeddings(context)
        embeds = embeds.mean(dim=0)

        return self.linear(embeds)


# Initialize model
model = CBOW(len(vocab), 10)

loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.SGD(
    model.parameters(), lr=0.01
)

In [6]:
# 4. Train Model
for epoch in range(100):

    for context, target in data:

        inputs = torch.tensor(
            [word_to_ix[w] for w in context]
        )

        label = torch.tensor(
            [word_to_ix[target]]
        )

        output = model(inputs)

        loss = loss_fn(
            output.unsqueeze(0), label
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


In [7]:
# 5. Prediction
context = ["I", "learning"]

inputs = torch.tensor(
    [word_to_ix[w] for w in context]
)

with torch.no_grad():

    output = model(inputs)

    predicted = ix_to_word[
        output.argmax().item()
    ]

print("Context words:", context)
print("Predicted word:", predicted)

Context words: ['I', 'learning']
Predicted word: and
